In [ ]:
import os
from dotenv import load_dotenv
import numpy as np
from typing import List, Dict
import warnings
warnings.filterwarnings("ignore")

#Langchain core import
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage


#Langchain specific import
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, ChatHuggingFace
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

load_dotenv()


True

Data Ingestion and Preprocessing

In [2]:
sample_document = [
    Document(
        page_content = """The attention mechanism is a transformative concept in deep learning, enabling models to selectively focus on the most
          relevant parts of input data. Inspired by human selective focus, it has significantly enhanced tasks like machine 
          translation, image recognition, and speech processing.""",
        metadata = {"source": "Attention Mechanism Introduction" , "page" : 1, "topic": "AI"}

    ),
    Document(
        page_content = """How the Attention Mechanism Works
    The attention mechanism operates by assigning weights to different elements of the input data, allowing the model to prioritize critical 
 information. This process involves several steps: Input Encoding: The input data is transformed into a format the model can process, 
 creating representations of the data.
 Query, Key, and Value Creation: The input is split into query, key, and value vectors. 
 Queries represent what the model is searching for, keys represent the data's features, and values hold the actual data.
Similarity Computation: The model calculates the similarity between the query and each key using methods like dot products or cosine similarity.
Attention Weights: Similarity scores are passed through a softmax function to compute attention weights, indicating the importance of each input element.
Context Vector: A weighted sum of the values is computed using the attention weights, forming a context vector that captures the most relevant information.
Integration: The context vector is combined with the model's current state to guide predictions.
This process is repeated iteratively, allowing the model to focus on different parts of the input sequence at each step.""",
        metadata = {"source": "Attention Mechanism Working" , "page" : 2, "topic": "AI"}
    ),

Document(
    page_content= """Applications of the Attention Mechanism
The attention mechanism has revolutionized various domains:
Machine Translation: It improves translation accuracy by focusing on relevant words in the source sentence for each target word.
Text Summarization: It helps models extract key information to generate concise summaries.
Image Captioning: It enables models to focus on specific parts of an image while generating descriptive captions.
Sentiment Analysis: It identifies critical words or phrases that influence sentiment classification.
Medical Imaging: It highlights regions of interest in medical scans for accurate diagnosis.""",
metadata={"source": "Attention Mechanism Applications", "page": 3, "topic": "AI"}
)

]

In [3]:
#Text splitting
text_Splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 100,
    separators= [" "],
    length_function = len
)

In [4]:
#Split the document into chunks
split_docs = text_Splitter.split_documents(sample_document)
print(f"Number of chunks created: {len(split_docs)}")

Number of chunks created: 6


In [5]:
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

In [6]:
#initialize the Hugging face embedding models

embedding_model = HuggingFaceEmbeddings(
    model_name = "Qwen/Qwen3-Embedding-0.6B",
)
#Creating a sample embedding
sample_embedding = embedding_model.embed_query("What is RAG")
print(sample_embedding)


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 1128.21it/s]


[-0.01153564453125, -0.005828857421875, -0.010498046875, -0.1083984375, 0.01806640625, 0.0186767578125, -0.054443359375, 0.031494140625, -0.0003681182861328125, 0.0162353515625, -0.048828125, -0.03662109375, 0.06298828125, -0.0103759765625, -0.055419921875, 0.08642578125, -0.07568359375, 0.06689453125, -0.01519775390625, -0.01123046875, -0.0186767578125, 0.05029296875, 0.0244140625, 0.09033203125, -0.033447265625, 0.020263671875, -0.033203125, 0.0198974609375, -0.002716064453125, -0.032958984375, -0.0174560546875, -0.005279541015625, -0.0517578125, -0.0224609375, -0.032958984375, -0.01446533203125, -0.0257568359375, -0.028076171875, 0.0269775390625, 0.020263671875, -0.0194091796875, 0.027587890625, 0.052978515625, -0.0498046875, -0.0302734375, -0.05224609375, 0.052490234375, 0.005645751953125, -0.003997802734375, 0.00885009765625, -0.0458984375, -0.046142578125, -0.037841796875, -0.00147247314453125, -0.00390625, -0.051513671875, 0.021728515625, 0.0225830078125, -0.007049560546875, -0.

In [8]:
#Batch embeddings
texts=["AI","MAchine learning","Deep Learning","Neural Network"]
batch_embeddings = embedding_model.embed_documents(texts)
print(batch_embeddings)

[[-0.0294189453125, 0.0281982421875, -0.00946044921875, -0.0869140625, -0.00860595703125, 0.00653076171875, -0.032470703125, 0.04833984375, -0.00335693359375, -0.0194091796875, 0.0164794921875, -0.05859375, 0.052001953125, -0.0106201171875, -0.049072265625, 0.059814453125, -0.11474609375, 0.040283203125, 0.0162353515625, -0.024658203125, -0.016357421875, 0.051025390625, -0.0157470703125, 0.09619140625, -0.0439453125, -0.00970458984375, -0.052978515625, 0.1083984375, -0.02783203125, -0.020263671875, -0.040283203125, 0.0306396484375, -0.0164794921875, -0.00445556640625, -0.032470703125, -0.014404296875, -0.00119781494140625, -0.00970458984375, -0.034912109375, 0.0556640625, 0.01513671875, -0.0123291015625, 0.02490234375, -0.0211181640625, -0.0194091796875, -0.01513671875, 0.1044921875, -0.0167236328125, -0.015380859375, -0.0303955078125, 0.002593994140625, -0.052734375, -0.0023193359375, 0.0027008056640625, 0.007232666015625, 0.01251220703125, 0.0234375, -0.0458984375, -0.02734375, -0.00

In [10]:
#Compare embeddings using cosine similarity

def compare_embeddings(text1: str, text2:str):
    """Compare cosine similairity between two texts"""
    emb1 = np.array(embedding_model.embed_query(text1))
    emb2 = np.array(embedding_model.embed_query(text2))

    #Calculate the cosine similarity
    similarity = np.dot(emb1, emb2)/(np.linalg.norm(emb1) * np.linalg.norm(emb2))
    return similarity

In [11]:
print("Testing similarity search")
print(f"\n {compare_embeddings('What is ML', 'What is Python')}")

Testing similarity search

 0.6112847292496968


Create FAISS Vector store

In [12]:
vector_store = FAISS.from_documents(
    documents= split_docs,
    embedding= embedding_model
)

In [13]:
print(f"Total number of documents in the vector store: {vector_store.index.ntotal}")

Total number of documents in the vector store: 6


In [14]:
#Saving vector store for later use
vector_store.save_local("faiss_index")
print("Vector store saved to faiss_index directory")

Vector store saved to faiss_index directory


In [15]:
#Load vector store
loaded_vector = FAISS.load_local(
    "faiss_index",
    embedding_model,
    allow_dangerous_deserialization= True
)

In [18]:
#Similarity search
query = "What are the implications of Ai in the lifes of humans"

ans = loaded_vector.similarity_search(query, k=3)
print(ans[0].page_content)

generating descriptive captions.
Sentiment Analysis: It identifies critical words or phrases that influence sentiment classification.
Medical Imaging: It highlights regions of interest in medical scans for accurate diagnosis.


In [19]:
#Similarity search with score
ans_with_score = loaded_vector.similarity_search_with_score(query, k=3)

In [21]:
for doc, score in ans_with_score:
    print(f"\nScore: {score:.3f}")
    print(f"Source: {doc.metadata['source']}")
    print(f"Content preview: {doc.page_content[:100]}...")


Score: 0.982
Source: Attention Mechanism Applications
Content preview: generating descriptive captions.
Sentiment Analysis: It identifies critical words or phrases that in...

Score: 1.274
Source: Attention Mechanism Introduction
Content preview: The attention mechanism is a transformative concept in deep learning, enabling models to selectively...

Score: 1.277
Source: Attention Mechanism Applications
Content preview: Applications of the Attention Mechanism
The attention mechanism has revolutionized various domains:
...


In [24]:
#Search with meta data filtering
filter_dict = {"page": 1}
filtered_result = loaded_vector.similarity_search(

    query,
    k=3,
    filter= filter_dict
)
filtered_result

[Document(id='10235049-49c5-4c73-8b6d-b275021a697f', metadata={'source': 'Attention Mechanism Introduction', 'page': 1, 'topic': 'AI'}, page_content='The attention mechanism is a transformative concept in deep learning, enabling models to selectively focus on the most\n          relevant parts of input data. Inspired by human selective focus, it has significantly enhanced tasks like machine \n          translation, image recognition, and speech processing.')]

Build RAG channel with LECL
